# Test Suite: Final Protein Ranking

Unit tests for the Alzheimer's Research GNN project.

In [ ]:
"""
Unit tests for final protein ranking module.
"""

import numpy as np
import pandas as pd
import pytest
import networkx as nx
from pathlib import Path

from src.analysis.final_ranking import ProteinRankingIntegrator


@pytest.fixture
def sample_explainability_df():
    """Create sample explainability data."""
    proteins = [f"PROT_{i}" for i in range(100)]
    data = {
        "protein": proteins,
        "mean_attr": np.random.random(100),
        "median_attr": np.random.random(100),
        "std_attr": np.random.random(100) * 0.1,
        "bootstrap_frequency_top100": np.random.randint(0, 101, 100),
    }
    return pd.DataFrame(data)


@pytest.fixture
def sample_graph():
    """Create sample protein interaction graph."""
    G = nx.barabasi_albert_graph(100, 3) # Network with 100 nodes
    proteins = [f"PROT_{i}" for i in range(100)]
    mapping = {i: p for i, p in enumerate(proteins)}
    return nx.relabel_nodes(G, mapping)


@pytest.fixture
def temp_data(tmp_path, sample_explainability_df, sample_graph):
    """Create temporary test data."""
    # Save explainability CSV
    csv_path = tmp_path / "test_attributions.csv"
    sample_explainability_df.to_csv(csv_path, index=False)

    # Save graph
    graph_path = tmp_path / "test_graph.graphml"
    nx.write_graphml(sample_graph, graph_path)

    return tmp_path, csv_path, graph_path


def test_integrator_initialization(temp_data):
    """Test ProteinRankingIntegrator initialization."""
    tmp_path, csv_path, graph_path = temp_data

    integrator = ProteinRankingIntegrator(
        results_dir=tmp_path / "results",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )

    assert integrator.cohort == "TEST"
    assert len(integrator.explainability_df) == 100
    assert len(integrator.graph) == 100


def test_network_centrality(temp_data):
    """Test network centrality computation."""
    tmp_path, csv_path, graph_path = temp_data

    integrator = ProteinRankingIntegrator(
        results_dir=tmp_path / "results",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )

    centrality = integrator.compute_network_centrality()

    assert len(centrality) == 100
    assert all(0 <= v <= 1 for v in centrality.values())
    assert all(isinstance(v, (int, float)) for v in centrality.values())


def test_normalize_values(temp_data):
    """Test value normalization."""
    tmp_path, csv_path, graph_path = temp_data

    integrator = ProteinRankingIntegrator(
        results_dir=tmp_path / "results",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )

    values = {f"PROT_{i}": i for i in range(10)}
    normalized = integrator.normalize_values(values)

    assert len(normalized) == 10
    assert min(normalized.values()) >= 0
    assert max(normalized.values()) <= 1
    assert np.isclose(min(normalized.values()), 0)
    assert np.isclose(max(normalized.values()), 1)


def test_composite_score_computation(temp_data):
    """Test composite score computation."""
    tmp_path, csv_path, graph_path = temp_data

    integrator = ProteinRankingIntegrator(
        results_dir=tmp_path / "results",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )

    # Create test case with all 100 proteins
    proteins = [f"PROT_{i}" for i in range(100)]
    gnn = {p: np.random.random() for p in proteins}
    centrality = {p: np.random.random() for p in proteins}
    bootstrap = {p: np.random.randint(0, 101) for p in proteins}

    scores = integrator.compute_composite_scores(gnn, centrality, bootstrap)

    # Should have score for all proteins in graph (100)
    assert len(scores) == 100
    assert all(v >= 0 for v in scores.values())


def test_generate_ranking(temp_data):
    """Test ranking generation."""
    tmp_path, csv_path, graph_path = temp_data

    integrator = ProteinRankingIntegrator(
        results_dir=tmp_path / "results",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )

    ranking_df = integrator.generate_ranking()

    assert len(ranking_df) == 100
    assert "protein" in ranking_df.columns
    assert "composite_score" in ranking_df.columns
    assert "gnn_importance" in ranking_df.columns
    assert "centrality" in ranking_df.columns
    assert "stability" in ranking_df.columns
    # Check that scores are in descending order
    assert ranking_df["composite_score"].is_monotonic_decreasing


def test_protein_to_gene_mapping(temp_data):
    """Test protein to gene mapping."""
    tmp_path, csv_path, graph_path = temp_data

    integrator = ProteinRankingIntegrator(
        results_dir=tmp_path / "results",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )

    ranking_df = integrator.generate_ranking()
    ranking_df = integrator.map_proteins_to_genes(ranking_df)

    assert "gene" in ranking_df.columns
    assert len(ranking_df) == 100
    assert len(ranking_df.columns) == 6 # protein, gene, composite_score, gnn, centrality, stability


def test_top_k_subgraph(temp_data):
    """Test subgraph extraction."""
    tmp_path, csv_path, graph_path = temp_data

    integrator = ProteinRankingIntegrator(
        results_dir=tmp_path / "results",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )

    ranking_df = integrator.generate_ranking()
    subgraph = integrator.get_top_k_subgraph(ranking_df, k=20)

    assert len(subgraph.nodes()) <= 100
    assert len(subgraph.edges()) >= 0


def test_stability_weight_formula():
    """Test stability weight formula: log(1 + freq)."""
    from src.analysis.final_ranking import ProteinRankingIntegrator

    # Test formula: log(1 + bootstrap_freq) / log(101)
    # At freq=0: log(1)/log(101) = 0
    # At freq=100: log(101)/log(101) = 1
    # At freq=50: log(51)/log(101) ≈ 0.83

    for freq in [0, 25, 50, 75, 100]:
        weight = np.log(1 + freq) / np.log(101)
        assert 0 <= weight <= 1


def test_ranking_deterministic(temp_data):
    """Test that ranking is deterministic."""
    tmp_path, csv_path, graph_path = temp_data

    # Run twice
    integrator1 = ProteinRankingIntegrator(
        results_dir=tmp_path / "results1",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )
    ranking1 = integrator1.generate_ranking()

    integrator2 = ProteinRankingIntegrator(
        results_dir=tmp_path / "results2",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )
    ranking2 = integrator2.generate_ranking()

    # Top proteins should be same (allowing for floating point differences)
    assert ranking1.head(10)["protein"].equals(ranking2.head(10)["protein"])


def test_composite_score_formula(temp_data):
    """Test composite score formula: GNN × centrality × (1 + stability_weight)."""
    tmp_path, csv_path, graph_path = temp_data

    integrator = ProteinRankingIntegrator(
        results_dir=tmp_path / "results",
        explainability_csv=csv_path,
        graph_file=graph_path,
        cohort="TEST",
    )

    # Create controlled test case with all 100 proteins
    proteins = [f"PROT_{i}" for i in range(100)]
    gnn = {p: 0.5 for p in proteins}
    centrality = {p: 0.5 for p in proteins}
    bootstrap = {p: 50 for p in proteins}

    scores = integrator.compute_composite_scores(gnn, centrality, bootstrap)

    # All scores should be positive (normalized input × normalized input × stability_weight)
    assert all(score >= 0 for score in scores.values())


if __name__ == "__main__":
    pytest.main([__file__, "-v"])
